In [1]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np

In [54]:
class recipe_fetch():
    def __init__(self):
        self.conn = sql.connect_pc()
        self.cursor = self.conn.cursor()


    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.conn.close()
        self.conn = None

    def __del__(self):
        if self.conn:
            self.conn.close()



    def id_range(self):
        query = """
        SELECT recipe_id FROM recipes
        WHERE name LIKE '%Charm%'
        OR name LIKE '%Vial%'
        OR name LIKE '%Ingot%'
        OR name LIKE '%Powder%'
        OR name LIKE '%Vial%'
        OR name LIKE '%Ghostdust%'
        OR name LIKE '%Gold Coin%'
        OR name LIKE '%Potion%'
        OR name LIKE '%Fangs%'
        OR name LIKE '%Campfire%'
        OR name LIKE '%Woodsman%'
        OR name LIKE '%Intricate%'
        OR name LIKE '%LockPicks%'
        """
        # query = """
        # SELECT recipe_id FROM recipes
        # """
        self.cursor.execute(query)
        return self.cursor.fetchall()

    def craftable(self, recipe_id):
        query = f"""
        SELECT amount, rarity, name FROM recipes
        WHERE recipe_id = {recipe_id}
        """
        self.cursor.execute(query)
        return self.cursor.fetchall() 

    def ingredients(self, recipe_id):
        query = f"""
        SELECT amount, rarity, name FROM ingredients
        WHERE recipe_id = {recipe_id}
        """
        self.cursor.execute(query)
        return self.cursor.fetchall()



In [43]:
db = darkerdb()
db.get_url('Gold Coin Chest','Unique')
fetch = db.price_fetch()
fetch

(12000,
 13000,
 14500,
 14444,
 14500,
 14750,
 14999,
 17000,
 14999,
 15000,
 11111,
 11111,
 13000,
 14500,
 14900,
 15000,
 15000,
 15998,
 16699,
 15999,
 15555,
 16700,
 16000,
 19800,
 16999,
 17000,
 17500,
 17500,
 18000,
 18660,
 18555,
 18887,
 18888,
 18888,
 18888,
 18888,
 18888,
 17315,
 18000,
 18500,
 17315,
 18500,
 18500,
 17750,
 17777,
 18000,
 18222,
 18255,
 18255,
 18255)

In [29]:
class darkerdb():
    def __init__(self):
        self.ses = requests.Session()

    def get_url(self, name, rarity):
        from_date = (datetime.now(timezone.utc) - timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
        self.url = f"https://api.darkerdb.com/v1/market?item={name.replace('\'', "’")}&rarity={rarity}&from={from_date}&limit=50&has_sold=0"
    def price_fetch(self):
        fetch = self.ses.get(self.url, timeout=10)
        price = tuple(item['price_per_unit'] for item in fetch.json()['body'])
        return (price)

In [62]:
r = recipe_fetch()
db = darkerdb()
for amount, rarity, name in r.ingredients(9):
        # print(amount, rarity, name)
        db.get_url(name, rarity)
        price_list = db.price_fetch()
        if name != 'Gold Coin':
                print(f"{name} ({round(np.nanmean(price_list)) * amount}) {price_list}")
        else: print(f"{name} ({amount})")

Obsidian Ore (242) (119.67, 119.67, 119.67, 119.67, 119.83, 88.8, 120, 120, 120, 120, 120.83, 120, 120, 117.6, 121.33, 121.33, 121.33, 126.5, 152, 126.67, 126.67)


In [63]:
db = darkerdb()


with recipe_fetch() as recipe:
    output = [('recipe_id', 'rarity', 'name', 'net', 'r_avg', 'i_avg', 'q_sold')]    
    for item in recipe.id_range():
        r_avg = 0
        i_avg = 0
        # print(item)
        
        for r_amount, r_rarity, r_name in recipe.craftable(item[0]):
            print(r_name, end="\r")
            db.get_url(r_name, r_rarity)
            price_list = db.price_fetch()
            q_sold = len(price_list)
            if price_list == (): pass
            else: r_avg += round(np.nanmean(price_list)) * r_amount

        for i_amount, i_rarity, i_name in recipe.ingredients(item[0]):
            if i_name != 'Gold Coin':
                db.get_url(i_name, i_rarity)
                price_list = db.price_fetch()
                if not price_list: pass
                else: i_avg += round(np.nanmean(price_list)) * i_amount
            else:
                i_avg += i_amount

        output.append((item[0], r_rarity, r_name, r_avg-i_avg, r_avg, i_avg, q_sold))
        # print(f"{item[0]} {r_rarity:<10}  {r_name:<35} net:{(r_avg-i_avg):>6}  r:{r_avg:>4}  i:{i_avg:>5}  q:{amount_sold:>5}")


In [65]:
output[1:] = sorted(output[1:], key=lambda x: x[3], reverse=True)
for x in output:
    print(f"{x[0]:<10} {x[1]:<10} {x[2]:<35} {(x[3]):>8} {x[4]:>8} {x[5]:>8} {x[6]:>8}")

recipe_id  rarity     name                                     net    r_avg    i_avg   q_sold
105        Rare       Charm of Fortune                         763     2758     1995       10
26         Rare       Potion of Water Breathing                483      483        0        9
24         Epic       Potion of Clarity                        429     1362      933       10
23         Rare       Potion of Clarity                        345     1137      792       16
81         Common     Iron Ingot                               262      544      282       13
195        Common     Iron Ingot                               259      544      285       13
85         Epic       Tidestone Ingot                          253      559      306        3
199        Epic       Tidestone Ingot                          253      559      306        3
12         Epic       Potion of Healing                        232      618      386       49
6          Epic       Gold Powder                           